# Newsroom Data Preprocessing

This notebook handles data loading, cleaning, and preparation for the Newsroom analysis.


In [1]:
import pandas as pd
import numpy as np
import ast
import re

## Load Raw Data


In [5]:
# Load raw data
df = pd.read_csv("../data/original_data/newsroom_judged.csv")

print("=" * 50)
print("RAW DATA")
print("=" * 50)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/original_data/newsroom_judged.csv'

In [ ]:
# Keep only relevant columns
columns_to_keep = [
    # Human Judgements
    'informativeness_scores', 'relevance_scores', 'fluency_scores', 'coherence_scores',
    # GPT
    'GPT_informativeness_as_a_judge', 'GPT_relevance_as_a_judge',
    'GPT_fluency_as_a_judge', 'GPT_coherence_as_a_judge',
    # LLAMA
    'LLAMA_informativeness_as_a_judge', 'LLAMA_relevance_as_a_judge',
    'LLAMA_fluency_as_a_judge', 'LLAMA_coherence_as_a_judge',
    # MISTRAL
    'MISTRAL_informativeness_as_a_judge', 'MISTRAL_relevance_as_a_judge',
    'MISTRAL_fluency_as_a_judge', 'MISTRAL_coherence_as_a_judge'
]

df = df.loc[:, columns_to_keep]
print(f"Shape after keeping relevant columns: {df.shape}")
print(f"Columns: {list(df.columns)}")


## Process Human Scores

Convert human score strings to lists and verify all have 3 judges.


In [ ]:
# Get human score columns
human_scores = [col for col in df.columns if 'scores' in col]

# Safely evaluate string representations of lists
for col in human_scores:
    df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# Verify all lists have 3 judges
print("Checking human score list lengths:")
for col in human_scores:
    lengths = df[col].apply(len)
    print(f"\n{col}:")
    print(lengths.value_counts())
    all_same_length = lengths.nunique() == 1
    print(f"All lists same length? {all_same_length}")


In [ ]:
# Break up human judgements into separate columns (human_#1, human_#2, human_#3)
for col in human_scores:
    metric_name = col.split('_')[0]
    for i in range(1, 4):
        new_col_name = f'{metric_name}_human_#{i}'
        df[new_col_name] = df[col].apply(lambda x: x[i-1])

# Drop original list columns
df.drop(columns=human_scores, inplace=True)

print(f"Shape after expanding human scores: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(3)


## Clean LLM Scores

Extract numerical ratings from LLM text outputs.


In [ ]:
def extract_valid_rating(series: pd.Series) -> pd.Series:
    """
    Extract leading number from strings in a Series,
    keeping only values between 1 and 5 inclusive.
    Non-matching or out-of-range values are returned as NaN.

    Parameters
    ----------
    series : pd.Series

    Returns
    -------
    pd.Series
        Series of integers in [1, 5] or NaN.
    """
    # Pattern matches optional whitespace + number with optional decimal part
    pattern = re.compile(r'^\s*(\d+(?:\.\d+)?)')

    values = []
    for item in series:
        text = str(item)
        match = pattern.match(text)
        if match:
            value = float(match.group(1))
            if 1 <= value <= 5:
                values.append(value)
            else:
                values.append(np.nan)
        else:
            values.append(np.nan)

    return pd.Series(values, index=series.index)


In [ ]:
# Get model columns
model_columns = [col for col in df.columns if 'as_a_judge' in col]

# Drop rows where any model column contains 'Evaluation Error'
mask = df[model_columns].apply(lambda row: row.astype(str).str.contains("Evaluation Error")).any(axis=1)
rows_before = len(df)
df = df[~mask]
print(f"Dropped {rows_before - len(df)} rows with 'Evaluation Error'")

# Drop NaN values
rows_before = len(df)
df = df.dropna()
print(f"Dropped {rows_before - len(df)} rows with NaN values")

# Apply cleaning and convert to int
for col in model_columns:
    df[col] = extract_valid_rating(df[col])
    df[col] = df[col].round().astype("Int64")

print(f"\nFinal shape: {df.shape}")
df.head(3)


In [ ]:
# Check for remaining null values
print("Null values per column:")
print(df.isnull().sum())


## Save Cleaned Data


In [ ]:
# Save cleaned dataset
df.to_csv("../data/newsroom_judged_cleaned.csv", index=False)
print(f"✅ Saved cleaned dataset to ../data/newsroom_judged_cleaned.csv")
print(f"   Shape: {df.shape}")

print()
print("=" * 50)
print("PREPROCESSING COMPLETE")
print("=" * 50)
print()
print("Output file:")
print("  - ../data/newsroom_judged_cleaned.csv")
